# Silver: shipments
Zero-quantity shipments are **valid** (the supplier shipped nothing) and stay in Silver: quarantining them would hide
the worst deliveries and bias supplier fill-rate upward. Only impossible values (negative qty, arrival before ship) are quarantined.

In [0]:
%run ./_common

In [0]:
NB = "22_silver_shipments"
SRC, TGT, QUAR = f"{CAT}.bronze.shipments", f"{CAT}.silver.shipments", f"{CAT}.silver.shipments_quarantine"
CP = f"{CKPT}/silver_shipments"

cols = """shipment_id STRING, po_id STRING, ship_date DATE, arrival_date DATE, qty_shipped INT,
          carrier_code STRING, _source_file STRING, _ingested_at TIMESTAMP"""
spark.sql(f"CREATE TABLE IF NOT EXISTS {TGT} ({cols}, _silver_at TIMESTAMP) CLUSTER BY (arrival_date)")
spark.sql(f"CREATE TABLE IF NOT EXISTS {QUAR} ({cols}, _rescued_data STRING, _dq_reason STRING, _quarantined_at TIMESTAMP)")

def process(batch_df, batch_id):
    s = batch_df.sparkSession
    if "carrier_code" not in batch_df.columns:                      # column only exists after the schema drift
        batch_df = batch_df.withColumn("carrier_code", F.lit(None).cast("string"))
    df = batch_df.select(
        "shipment_id", "po_id",
        F.to_date("ship_date").alias("ship_date"), F.to_date("arrival_date").alias("arrival_date"),
        F.col("qty_shipped").cast("int").alias("qty_shipped"),
        F.col("carrier_code").cast("string").alias("carrier_code"),
        "_source_file", "_ingested_at", "_rescued_data")
    reason = (F.when(F.col("_rescued_data").isNotNull(), "rescued_data")
               .when(F.col("shipment_id").isNull() | F.col("po_id").isNull(), "missing_key")
               .when(F.col("qty_shipped").isNull() | (F.col("qty_shipped") < 0), "invalid_qty")
               .when(F.col("arrival_date") < F.col("ship_date"), "arrives_before_shipped"))
    df = df.withColumn("_dq_reason", reason)
    (df.filter("_dq_reason IS NOT NULL").withColumn("_quarantined_at", F.current_timestamp())
       .select(*s.table(QUAR).columns).write.mode("append")
       .option("txnAppId", "silver_shipments_quarantine").option("txnVersion", batch_id)
       .saveAsTable(QUAR))
    w = Window.partitionBy("shipment_id").orderBy("_ingested_at")
    good = (df.filter("_dq_reason IS NULL")
              .withColumn("_rn", F.row_number().over(w)).filter("_rn = 1")
              .withColumn("_silver_at", F.current_timestamp())
              .select(*s.table(TGT).columns))
    (DeltaTable.forName(s, TGT).alias("t")
        .merge(good.alias("s"), "t.shipment_id = s.shipment_id")
        .whenNotMatchedInsertAll().execute())

before = table_count(TGT)
q = (spark.readStream.option("maxBytesPerTrigger", "64m").table(SRC)
       .writeStream.foreachBatch(process)
       .option("checkpointLocation", CP)
       .trigger(availableNow=True).start())
q.awaitTermination()
print(f"{table_count(TGT) - before:,} shipments added (total {table_count(TGT):,}); quarantined {table_count(QUAR):,}")

In [0]:
b, s, qn = table_count(SRC), table_count(TGT), table_count(QUAR)
dq_check("shipments_reconcile_to_bronze", s + qn, s + qn == b, f"{b} (bronze rows)")
zero = spark.table(TGT).where("qty_shipped = 0").count()
dq_check("zero_qty_shipments_kept", zero, True, "informational", hard=False)
dq_finish(NB)